# Wireless Networking and Mobile Computing

### Prerequisites
This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install matplotlib pandas scipy --quiet

**Imports**

In [1]:
# --- Imports: libraries used in this notebook ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

**Declarations**

Set the variables below and run the generation cell.  

- The datasets will be loaded from the listed CSV file paths (each expected to contain either one column of direct numeric data or a jemula result csv file).  
- `confidence_level` defines the width of the confidence interval (e.g., 0.95 for 95%).  


In [ ]:
# --- Declarations: editable variables for students ---
csv_files = [
    './dataset_A.csv',
    ...
]

confidence_level = ...

### 1. Introduction

#### 1.1 Definition

Confidence intervals (CIs) express **uncertainty around an estimated parameter**, such as the sample mean.  
For a 95% confidence interval, if we repeated the experiment infinitely many times, about 95% of intervals would contain the true population mean.

$$
\bar{x} \pm z \frac{s}{\sqrt{n}},
$$

where
- $\bar{x}$ is the sample mean  
- $s$ is the sample standard deviation  
- $n$ is the sample size  
- $z$ is the critical value (≈ 1.96 for 95% CI under normal assumption)

In [ ]:
# --- Load Data ---
datasets = {}
for path in csv_files:
    name = Path(path).stem
    try:
        preview = pd.read_csv(path, nrows=5)
        if preview.shape[1] == 1:
            df = pd.read_csv(path)
            data = df.iloc[:, 0].dropna().to_numpy(dtype=float)
        else:
            df = pd.read_csv(path, skiprows=2)
            data = np.column_stack((
                df.iloc[:, 0].to_numpy(dtype=float) / 1000,
                df.iloc[:, 6].to_numpy(dtype=float)
            ))
        datasets[name] = data
        print(f"Loaded {name}: {len(df)} samples ({df.shape[1]} cols)")
    except Exception as e:
        print(f"Error loading {path}: {e}")

In [ ]:
# --- Compute Confidence Intervals ---
for name, data in datasets.items():
    n = len(data)
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    se = std / np.sqrt(n)
    ci_low, ci_high = stats.t.interval(confidence_level, df=n-1, loc=mean, scale=se)

    print(f"\nDataset: {name}")
    print(f" - n:        {n}")
    print(f" - Mean:     {mean:.4f}")
    print(f" - StdDev:   {std:.4f}")
    print(f" - CI:       [{ci_low:.4f}, {ci_high:.4f}]")

In [ ]:
# --- Plot Confidence Intervals per Dataset ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.set_facecolor((0.15, 0.15, 0.22))
colors = ['gold', 'lime', 'cyan']

for i, (name, row) in enumerate(df_results.iterrows()):
    ax.errorbar(i, row['Mean'],
                yerr=[[row['Mean'] - row['CI_Lower']], [row['CI_Upper'] - row['Mean']]],
                fmt='o', color=colors[i % len(colors)], capsize=5, label=name)

ax.set_xticks(range(len(df_results)))
ax.set_xticklabels(df_results['Dataset'])
ax.set_title(f"{int(confidence_level*100)}% Confidence Intervals for Sample Means")
ax.set_xlabel('Dataset')
ax.set_ylabel('Value')
ax.grid(True, alpha=0.3)
ax.legend(facecolor=(0.2, 0.2, 0.3), framealpha=1, edgecolor='white')
plt.tight_layout()
plt.show()

#### 1.2 Usage
Confidence intervals can be computed **whenever repeated or random samples** are available and the data represent independent observations from a population.  
They assume the sampling distribution of the mean is approximately normal, which is valid either when the population itself is normal or when the **sample size is sufficiently large** (Central Limit Theorem).

#### 1.3 Interpretation
- The mean gives a point estimate of the central tendency.  
- The confidence interval quantifies uncertainty: a wider CI means more variability or fewer samples.  
- If two CIs overlap strongly there’s likely no statistically significant difference between datasets at the chosen confidence level.  
- Increasing sample size or lowering variability tightens the CI.